<a href="https://colab.research.google.com/github/mzaib1012/scada-substation-monitor/blob/main/SCADA_Substation_Monitor_Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install pymodbus==3.5.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: pymodbus
    Found existing installation: pymodbus 3.13.0
    Uninstalling pymodbus-3.13.0:
      Successfully uninstalled pymodbus-3.13.0


In [7]:
!pip install dash plotly pymodbus

In [8]:
import asyncio
import random
import threading
import time
from pymodbus.server import StartAsyncTcpServer
from pymodbus.datastore import ModbusSequentialDataBlock, ModbusSlaveContext, ModbusServerContext

# Setup Modbus Memory
store = ModbusSlaveContext(
    di=ModbusSequentialDataBlock(0, [0]*10),
    co=ModbusSequentialDataBlock(0, [0]*10),
    hr=ModbusSequentialDataBlock(0, [0]*10),
    ir=ModbusSequentialDataBlock(0, [0]*10)
)

# Instead of passing 'slave=' or 'slaves=',
# we pass the store as a dictionary or a direct object
# to the context initializer.
context = ModbusServerContext(store, single=True)

# 2. Simulation logic
def update_values(context):
    while True:
        voltage = int(230 + random.uniform(-5, 5))
        current = int(10 + random.uniform(-2, 2))
        switch_state = random.choice([0, 1])

        slave_context = context[0]
        slave_context.setValues(3, 0, [voltage, current])
        slave_context.setValues(1, 0, [switch_state])
        time.sleep(1)

# 3. Run Server
def run_server():
    asyncio.run(StartAsyncTcpServer(context, address=("0.0.0.0", 5020)))

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
sim_thread = threading.Thread(target=update_values, args=(context,), daemon=True)
sim_thread.start()

print("RTU Simulator is running successfully on port 5020...")

RTU Simulator is running successfully on port 5020...


In [10]:
from dash import Dash, html, dcc, Input, Output
import plotly.graph_objects as go
from pymodbus.client import ModbusTcpClient
import threading

# Connect to the local Modbus Server
def get_modbus_data():
    client = ModbusTcpClient('127.0.0.1', port=5020)
    client.connect()
    # Read 2 registers starting at 1 (Voltage, Current)
    regs = client.read_holding_registers(1, 2, slave=1)
    # Read 1 coil (Switch State)
    coil = client.read_coils(1, 1, slave=1)
    client.close()

    val = regs.registers if not regs.isError() else [0, 0]
    switch = coil.bits[0] if not coil.isError() else False
    return val[0], val[1], switch

app = Dash(__name__)

app.layout = html.Div([
    html.H1("Substation SCADA Dashboard"),
    dcc.Interval(id='interval', interval=1000, n_intervals=0),
    html.Div(id='live-display'),
    dcc.Graph(id='gauge-volt')
])

@app.callback(
    [Output('live-display', 'children'), Output('gauge-volt', 'figure')],
    [Input('interval', 'n_intervals')]
)
def update_dashboard(n):
    v, c, s = get_modbus_data()
    display = html.Div([
        html.P(f"Voltage: {v} V"),
        html.P(f"Current: {c} A"),
        html.P(f"Switchgear State: {'CLOSED' if s else 'OPEN'}")
    ])
    fig = go.Figure(go.Indicator(mode="gauge+number", value=v, title={'text': "Voltage"}))
    return display, fig

if __name__ == '__main__':
    # Updated: replaced app.run_server(mode='inline') with app.run()
    # In Google Colab, app.run() automatically detects the environment.
    app.run(debug=True, port=8050)

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: on
